In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 终端公共统计反馈：一次有界 GPU 试验入口
沿用已跑通的局部时间 warp 笔记本的 Drive、pip、子进程启动与结果路径习惯。此版只完成本地工程检查，尚未运行真实 Wan。运行下一入口会加载模型并占用 GPU，应由用户决定执行。3 臂 × 49 帧；124 Transformer、7 浮点 VAE、2 backward；1800 秒，零重试。
使用 GitHub/Colab 直接入口，不要求用户上传 ZIP；发布授权不等于 GPU 执行授权。


In [ ]:
from pathlib import Path
import sys, subprocess, json
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH = 'dev/生成端公共关系载体/二维图像统计-持续生成约束'
SOURCE = Path('/content/public_statistic_phase2_source')
if SOURCE.exists(): raise FileExistsError('Use a fresh runtime; preserve existing source')
# 获取已发布开发分支，并记录实际使用的提交。
subprocess.run(['git','init',str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'remote','add','origin',REPOSITORY_URL],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','--depth','1','origin',SOURCE_BRANCH],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)
print('Source:',subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip())


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','diffusers','transformers','accelerate','ftfy','sentencepiece','safetensors','huggingface_hub','numpy','Pillow'],check=True)
subprocess.run(['ffmpeg','-version'],check=True)
# Keep Colab CUDA PyTorch. Actual versions are recorded by the worker.


## 输入与预算
单一固定 prompt/seed，OFF1、OFF2 共享正常前缀并分别执行尾部，Y_MINUS 在 after47/48 作一次固定候选；接受仅看整段终端 loss 下降与 RGB RMSE ≤ 3/255。逐帧最坏值仅诊断。eta=10；单次更新≤0.001×前缀 RMS，累计≤0.002×前缀 RMS。模型和依赖实际版本仅记录。使用支持 BF16 的 CUDA GPU，建议单张 L4 24 GiB；CUDA 分配器上限22 GiB、进程树 RSS32 GiB。真实 backward 显存未验证，OOM 直接终止保留全部槽。


In [ ]:
CONFIG = SOURCE/'configs/public_luma_phase2_gpu.json'
print(CONFIG.read_text())
OUTPUT=Path('/content/drive/MyDrive/Video-WM/public-statistic-phase2/run01')
if OUTPUT.exists(): raise FileExistsError(str(OUTPUT))


In [ ]:
command=[sys.executable,'-m','experiments.public_statistic.run_phase2','--config',str(CONFIG),'--output',str(OUTPUT)]
result=subprocess.run(command,cwd=SOURCE)
print('launcher exit',result.returncode)
print((OUTPUT/'result.json').read_text() if (OUTPUT/'result.json').exists() else 'No result file')
result.check_returncode()


## 回传
将 OUTPUT 下 result.json、config.json、runtime.json、loaded_model.json、counts.json、execution.log、execution_exit.json 及全部三臂文件回传授权审计会话。失败/未执行行保留在固定147行内；不自动重试、不扩参数。浮点候选接受与 MP4 读回诊断分别记录。本入口没有独立科学验证，也不宣称稳健性或盲恢复成功。
